# Publication figures with SpatialBiologyToolkit

This runnable example creates two small synthetic ROIs in a new temporary directory. It does not read or change project datasets. Replace the `Dataset` paths and `adata` with your own assets afterwards.

Recipes contain layout and styling; dataset bindings contain sources and calibration. All panels share a single spatial view.


In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import anndata as ad
import tifffile
from PIL import Image
from SpatialBiologyToolkit import figures as F

root = Path(tempfile.mkdtemp(prefix="sbt-figures-example-"))
for name in ("imc", "masks", "external", "regions"):
    (root / name).mkdir()

rng = np.random.default_rng(42)
y, x = np.mgrid[:240, :320]
records, expression = [], []
for roi in ("ROI_1", "ROI_2"):
    (root / "imc" / roi).mkdir()
    mask = np.zeros((240, 320), dtype=np.uint32)
    cd3 = np.zeros(mask.shape, dtype=np.float32)
    cd68 = np.zeros_like(cd3)
    dna = np.zeros_like(cd3)
    for ident in range(1, 49):
        cx, cy = rng.integers(12, 308), rng.integers(12, 228)
        distance = (x-cx)**2 + (y-cy)**2
        mask[distance < 45] = ident
        signal = np.exp(-distance / 50)
        population = "T cells" if ident % 2 else "Macrophages"
        score = float(cx / 160 - 1 + rng.normal(0, .15))
        cd3 += signal * (4 if population == "T cells" else .3)
        cd68 += signal * (5 if population == "Macrophages" else .2)
        dna += signal * 3
        records.append(dict(ROI=roi, ObjectNumber=ident, X_loc=cx, Y_loc=cy,
                            population=population, score=score))
        expression.append([score])
    for name, pixels in (("CD3", cd3), ("CD68", cd68), ("DNA1", dna)):
        tifffile.imwrite(root / "imc" / roi / f"{name}.tif", pixels)
    tifffile.imwrite(root / "masks" / f"{roi}.tif", mask)
    labels = np.where(x < 160, 1, 2).astype(np.uint16)
    labels[(x-230)**2 + (y-120)**2 < 30**2] = 0
    tifffile.imwrite(root / "regions" / f"{roi}.tif", labels)
    external = np.stack([180 + x/8, 150 + y/5, 200 - x/8], axis=-1).astype(np.uint8)
    Image.fromarray(external[::2, ::2]).save(root / "external" / f"registered_{roi}.png")

adata = ad.AnnData(np.array(expression, dtype=np.float32),
                   obs=pd.DataFrame(records, index=[f"cell_{i}" for i in range(len(records))]),
                   var=pd.DataFrame(index=["synthetic_marker"]))
adata.obs["population"] = adata.obs["population"].astype("category")
data = F.Dataset(adata, imc_folder=root/"imc", mask_folder=root/"masks",
                 image_folders={"external": root/"external"},
                 label_folders={"regions": root/"regions"}, pixel_size_um=1.0)
data.describe()


## Define six panels

The external image is synthetic, at half the IMC pixel resolution. The annotation layer demonstrates independent tissue-label TIFFs and transparent holes. Limits here describe synthetic intensities, not real-data recommendations.


In [ ]:
figure = F.Figure(
    layout=(2, 3),
    crop=F.Crop(mode="center", size=(220, 180)),
    style=F.Style(panel_width_mm=55, panel_height_mm=45,
                  title_fontsize=11, legend_fontsize=8),
    panels=[
        F.Panel(row=0, col=0, title="IMC", letter="A", layers=[
            F.IMC(channels=[F.Channel("CD3", color="red", limits=(0, 4)),
                            F.Channel("CD68", color="green", limits=(0, 5)),
                            F.Channel("DNA1", color="blue", limits=(0, 3))])
        ], scale_bar=F.ScaleBar(length=50, unit="um")),
        F.Panel(row=0, col=1, title="Populations", letter="B", layers=[
            F.Populations(obs="population", mode="both", edgecolor="white",
                          colors={"T cells": "#e45756", "Macrophages": "#4c78a8"})]),
        F.Panel(row=0, col=2, title="External image", letter="C", layers=[F.Image(source="external")]),
        F.Panel(row=1, col=0, title="Cell score", letter="D", layers=[
            F.Values(value=F.obs("score"), scale=F.Scale(mode="fixed", limits=(-1.2, 1.2)), cmap="coolwarm")]),
        F.Panel(row=1, col=1, title="Tissue annotations", letter="E", layers=[
            F.Image(source="external"),
            F.LabelMask(source="regions", labels={1: "Region 1", 2: "Region 2"},
                        colors={1: "#54a24b", 2: "#eeca3b"}, opacity=.5)]),
        F.Panel(row=1, col=2, title="Expression", letter="F", layers=[
            F.Values(value=F.var("synthetic_marker"), cmap="magma")]),
    ],
)
prepared = figure.prepare(data)
with prepared.render("ROI_1", dpi=120) as preview:
    display(preview)


## Export all ROIs and review

Only final images and metadata are saved. Open `index.html` to compare ROIs. The SVGs have editable text and separate panel/layer groups; individual image pixels remain raster. Batch rendering closes figures automatically.


In [ ]:
manifest = figure.export_rois(data, root/"figures", formats=("png", "svg"))
print(root/"figures"/"index.html")
figure.save(root/"figure_recipe.yaml")
restored = F.Figure.load(root/"figure_recipe.yaml")
assert restored.model_dump() == figure.model_dump()


## Use annotations to select the shared view

A hotspot shows maximal activity, not necessarily representative tissue. Count, sum, mean and fraction have different meanings. Inspect the score and selected bounds alongside the output.


In [ ]:
figure.crop = F.Crop.hotspot(size=(180, 140), score=F.obs("score"), reducer="mean", min_cells=8)
with figure.render(data, "ROI_1", dpi=120) as hotspot:
    print(hotspot.metadata["view"])
    display(hotspot)

# Or select annotated area without requiring AnnData-linked objects:
figure.crop = F.Crop.hotspot(size=(180, 140), mask_source="regions", mask_labels=[2], reducer="fraction")
with figure.render(data, "ROI_1", dpi=120) as region:
    print(region.metadata["view"])
    display(region)


## Freeze selected coordinates for future figures

Saved bounds use `(x, y, width, height)` in reference pixels. Changing the panel composition or export DPI does not change these coordinates.


In [ ]:
views = {row["roi"]: row["view"]["bounds"] for row in manifest["results"] if row["status"] == "ok"}
figure.crop = F.Crop(mode="bounds", roi_bounds=views)
figure.save(root/"fixed_views.json")
print("All synthetic outputs:", root)
